# Artificial Intelligence Technology and Application

## Machine Learning Lab Guide - Student Version

Independent implementation prepared by **Sundetkhan Bekzat**.


# 1 Private Credit Default Prediction

This notebook keeps the lab objective but uses compact local examples so it can run without external datasets.


## 1.1 Imbalanced Classification
The generated dataset imitates rare default cases.


In [1]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=500, n_features=8, n_informative=5, weights=[0.84, 0.16], flip_y=0.02, random_state=19
)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=19)
print("train default rate:", round(y_train.mean(), 3))


train default rate: 0.171


## 1.2 Balanced Logistic Regression
Class weights reduce the bias toward the majority class.


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
model = LogisticRegression(max_iter=500, class_weight="balanced").fit(scaler.transform(X_train), y_train)
proba = model.predict_proba(scaler.transform(X_test))[:, 1]
pred = (proba >= 0.5).astype(int)
print("accuracy:", round(accuracy_score(y_test, pred), 3))
print("precision:", round(precision_score(y_test, pred), 3))
print("recall:", round(recall_score(y_test, pred), 3))
print("roc_auc:", round(roc_auc_score(y_test, proba), 3))


accuracy: 0.872
precision: 0.607
recall: 0.773
roc_auc: 0.893


## 1.3 Threshold Tuning
The threshold can be adjusted when recall is more important than raw accuracy.


In [3]:
for threshold in [0.35, 0.5, 0.65]:
    tuned = (proba >= threshold).astype(int)
    print(threshold, "recall=", round(recall_score(y_test, tuned), 3), "precision=", round(precision_score(y_test, tuned), 3))


0.35 recall= 0.818 precision= 0.486
0.5 recall= 0.773 precision= 0.607
0.65 recall= 0.682 precision= 0.75


## 1.4 Simple Hyperparameter Search
A small non-interactive search keeps the notebook fast.


In [4]:
best = None
for c_value in [0.1, 1.0, 10.0]:
    candidate = LogisticRegression(max_iter=500, class_weight="balanced", C=c_value)
    candidate.fit(scaler.transform(X_train), y_train)
    score = roc_auc_score(y_test, candidate.predict_proba(scaler.transform(X_test))[:, 1])
    best = max(best or (0, None), (score, c_value))
print("best C:", best[1], "auc:", round(best[0], 3))


best C: 0.1 auc: 0.896
